In [9]:
import numpy as np
import re

class traj(object):
    def __init__(self):
        self.frames = []

    @property
    def _nframes(self):
        return len(self.frames)

    def read_xyz_traj(self, filename, type="vmd"):
        file = open(filename, "r").readlines()
        if type == "vmd":
            total_atoms = int(file[0])
            i = 2
            while i < len(file) - 1:
                lines = []
                for j in range(total_atoms):
                    line = file[i + j].split()
                    if len(line) != 4:
                        print(f"Error in line {i+j}:{file[i+j]}")
                    line[0] = re.sub(r'\d+', '', line[0])
                    lines.append(line)
                self.frames.append(np.asarray(lines))
                i += total_atoms + 2
        else:
            raise TypeError(f"Invalid type: {type}")

    def get_unit_cell_traj(self, i, j, k, atoms_per_mol=162, supercell_dim=(3, 3, 3),
                           cell_params=(14.0135, 14.3092, 16.9072, 71.990, 85.828, 73.442)):
        """
        Extract trajectory of a specific unit cell (i, j, k) from a supercell trajectory.
        i, j, k: unit cell index, each in range [0, supercell_dim-1]
        """
        # Build supercell vectors from unit cell parameters
        a, b, c, alpha, beta, gamma = cell_params
        na, nb, nc = supercell_dim
        alpha, beta, gamma = np.radians(alpha), np.radians(beta), np.radians(gamma)

        # Supercell = unit cell * supercell_dim
        sa, sb, sc = a * na, b * nb, c * nc

        ax = sa
        bx = sb * np.cos(gamma)
        by = sb * np.sin(gamma)
        cx = sc * np.cos(beta)
        cy = sc * (np.cos(alpha) - np.cos(beta) * np.cos(gamma)) / np.sin(gamma)
        cz = np.sqrt(sc**2 - cx**2 - cy**2)

        cell_mat = np.array([[ax, 0, 0], [bx, by, 0], [cx, cy, cz]])
        inv_cell = np.linalg.inv(cell_mat.T)  # Cartesian -> fractional

        # Determine which residues belong to unit cell (i,j,k) using first frame
        frame0 = self.frames[0]
        coords = frame0[:, 1:4].astype(float)
        n_mol = coords.shape[0] // atoms_per_mol

        mol_indices = []  # indices of molecules in target unit cell
        for m in range(n_mol):
            # Geometric center of molecule
            mol_coords = coords[m * atoms_per_mol:(m + 1) * atoms_per_mol]
            center = mol_coords.mean(axis=0)

            # Fractional coordinates in supercell
            frac = inv_cell @ center
            frac = frac % 1.0  # wrap to [0, 1)

            # Unit cell index
            ci = int(frac[0] * na)
            cj = int(frac[1] * nb)
            ck = int(frac[2] * nc)

            if ci == i and cj == j and ck == k:
                mol_indices.append(m)

        print(f"Unit cell ({i},{j},{k}): {len(mol_indices)} molecules, "
              f"{len(mol_indices) * atoms_per_mol} atoms")

        # Extract matching atoms for all frames
        new_traj = traj()
        atom_indices = []
        for m in mol_indices:
            atom_indices.extend(range(m * atoms_per_mol, (m + 1) * atoms_per_mol))

        for frame in self.frames:
            new_traj.frames.append(frame[atom_indices])

        return new_traj

In [11]:
tem_traj = traj()
tem_traj.read_xyz_traj("test.xyz")
new_traj = tem_traj.get_unit_cell_traj(0, 0, 0)

Unit cell (0,0,0): 2 molecules, 324 atoms


In [13]:

from chemlab.util.file_system import qchem_file
import os

for i in range(10):
    tem_qc = qchem_file()
    tem_qc.read_from_file("ref.in")
    tem_qc.molecule.check = True
    tem_qc.molecule.carti = tem_traj.frames[i]
    tem_qc.generate_inp(f"test_{i:04d}.inp")